# পাঠ ০১ - AI এজেন্টদের পরিচিতি

**শুরু করতে ইচ্ছুকদের জন্য AI এজেন্ট** কোর্সের প্রথম পাঠে আপনাকে স্বাগতম!

একটি **AI এজেন্ট** হল এমন একটি প্রোগ্রাম যা একটি বড় ভাষার মডেল (LLM) কে তার যুক্তি চালনা ইঞ্জিন হিসাবে ব্যবহার করে এবং যা বাস্তব জগতে *কর্ম* নিতে পারে — API কল করা, ডাটাবেস প্রশ্ন করা, বা কোড চালানো — যাতে ব্যবহারকারীর পক্ষে একটি লক্ষ্য অর্জন করা যায়।

এই নোটবুকে আপনি আপনার প্রথম এজেন্ট তৈরি করবেন: একটি **ভ্রমণ এজেন্ট** যা ছুটির স্থানগুলোর সুপারিশ দেয়। পথে আপনি শিখবেন কীভাবে:

১. **Microsoft Agent Framework** ব্যবহার করে Microsoft Foundry Agent Service এর সাথে সংযুক্ত হওয়া।
২. এজেন্টকে একটি **টুল** প্রদান করা — একটি সাদামাটা পাইথন ফাংশন যা এটি কল করতে পারে।
৩. এজেন্ট চালানো এবং এর প্রতিক্রিয়া পরিদর্শন করা।
৪. এজেন্টের প্রতিক্রিয়া টোকেন-বাই-টোকেন স্ট্রিম করা।


## সেটআপ

এই নোটবুকটি চালানোর আগে নিশ্চিত করুন যে আপনার কাছে রয়েছে:

1. **একটি Microsoft Foundry প্রকল্প** যার মধ্যে একটি মোতায়েন করা চ্যাট মডেল (যেমন `gpt-4.1-mini`) আছে।
2. **Azure CLI তে লগ ইন করেছেন** — আপনার টার্মিনালে `az login` চালান।
3. **প্রয়োজনীয় পরিবেশ ভেরিয়েবলগুলি সেট করেছেন:**
   - `AZURE_AI_PROJECT_ENDPOINT` — আপনার Microsoft Foundry প্রকল্পের এন্ডপয়েন্ট।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার মোতায়েনকৃত মডেলের নাম।

নিচের সেলটি আপনার প্রয়োজনীয় পাইথন প্যাকেজগুলি ইনস্টল করবে।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## আপনার প্রথম এজেন্ট তৈরি করা

একটি এজেন্টের দুইটি জিনিস প্রয়োজন:

- **নির্দেশিকা** যা এটিকে বলে *সে কে* এবং *কিভাবে আচরণ করতে হবে* (একটি সিস্টেম প্রম্পট)।
- **টুলস** — Python ফাংশন যেগুলো `@tool` দিয়ে সাজানো থাকে এবং যেগুলো এজেন্ট তথ্য সংগ্রহ বা কাজ সম্পাদনের জন্য ব্যবহার করতে পারে।

নিচে আমরা একটি সাধারণ টুল সংজ্ঞায়িত করেছি যা জনপ্রিয় ছুটির গন্তব্যের একটি তালিকা প্রদান করে। যখন ব্যবহারকারী ভ্রমণের সুপারিশ চায়, তখন এজেন্ট এই টুলটি ব্যবহার করবে।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## স্ট্রিমিং প্রতিক্রিয়া

আরও ইন্টারেক্টিভ অভিজ্ঞতার জন্য আপনি এজেন্টের প্রতিক্রিয়া **স্ট্রিম** করতে পারেন। পুরো উত্তর অপেক্ষা করার পরিবর্তে, এজেন্ট তৈরি হওয়া সঙ্গে সঙ্গে টেক্সট চাঙ্ক সরবরাহ করে। এটি বিশেষ করে চ্যাট ইন্টারফেসে উপকারী যেখানে আপনি রিয়েল টাইমে আউটপুট প্রদর্শন করতে চান।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## সারাংশ

এই পাঠে আপনি শিখেছেন কিভাবে:

- **একটি প্রোভাইডার তৈরি করবেন** যা `FoundryChatClient` এর মাধ্যমে Microsoft Foundry Agent Service-এ সংযুক্ত হয়।
- **একটি টুল সংজ্ঞায়িত করবেন** `@tool` ডেকোরেটর ব্যবহার করে যাতে এজেন্ট আপনার পাইথন ফাংশনগুলি কল করতে পারে।
- **এজেন্ট চালাবেন** ইউজার মেসেজ দিয়ে এবং এর প্রতিক্রিয়া প্রিন্ট করবেন।
- **রিয়েল-টাইম আউটপুটের জন্য প্রতিক্রিয়া স্ট্রিম করবেন**।

পরবর্তী পাঠে আমরা এজেন্টিক ফ্রেমওয়ার্কগুলি আরও গভীরভাবে অনুসন্ধান করব এবং শিখব কিভাবে এজেন্টদের আরও শক্তিশালী টুল এবং বহু-ধাপ যুক্তি ক্ষমতা প্রদান করা যায়।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
